# Fase 1 — Adquisición y extracción de datos (Flujo A)

| Campo | Valor |
|---|---|
| **Rol líder** | Ingeniero de Datos |
| **Entrada** | `data/raw/pubchem_panama_cids.csv` (~235 compuestos PubChem + 20 MIDA) |
| **Salida principal** | `data/raw/chembl_panama_bioactivity.csv` |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase1_adquisicion_datos.md`](../docs/fases/fase1_adquisicion_datos.md) |
| **Script CLI** | `scripts/analisis_proyecto/fase1/extract_chembl_local.py` |
| **Comando Make** | `make chembl-extract` |

Pipeline completo de extracción ChEMBL.


## 0. Configuración


In [ ]:
import importlib
import sys
from pathlib import Path

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

import os

# Recargar módulos tras cambios (evita caché del kernel)
import src.analisis_proyecto.chembl_extract as chembl_extract
import src.analisis_proyecto.chembl_local as chembl_local
importlib.reload(chembl_local)
importlib.reload(chembl_extract)

from src.data.mida import MIDA_ACTIVE_INGREDIENTS
from src.analisis_proyecto.chembl_extract import (
    apply_quality_filters_from_config,
    build_bioactivity_table,
    build_mapping_table,
    load_chembl_config,
    load_corpus_compounds,
    resolve_corpus_mode,
    resolve_standard_types,
    summarize_extraction,
)
from src.analisis_proyecto.chembl_api import PCHEMBL_ACTIVE_THRESHOLD
from src.analisis_proyecto.chembl_local import db_info, ensure_db_exists

CHEMBL_CFG = load_chembl_config(ROOT / "config" / "config.yaml")
CHEMBL_BACKEND = os.environ.get("CHEMBL_BACKEND", CHEMBL_CFG.get("backend", "sqlite"))
CORPUS_MODE = os.environ.get("CHEMBL_CORPUS_MODE", resolve_corpus_mode(CHEMBL_CFG))
STANDARD_TYPES = resolve_standard_types(CHEMBL_CFG)
CHEMBL_DB = Path(os.environ.get("CHEMBL_DB_PATH", CHEMBL_CFG["db_path"]))
if not CHEMBL_DB.is_absolute():
    CHEMBL_DB = ROOT / CHEMBL_DB

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

CORPUS_CSV = ROOT / "data" / "raw" / "pubchem_panama_cids.csv"
OUT_DIR = ROOT / "data" / "raw"
FIG_DIR = ROOT / "outputs" / "chembl"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

COMPOUNDS_CSV = OUT_DIR / "chembl_corpus_compounds.csv"
MAPPING_CSV = OUT_DIR / "chembl_corpus_mapping.csv"
RAW_CSV = OUT_DIR / "chembl_panama_bioactivity_raw.csv"
CLEAN_CSV = OUT_DIR / "chembl_panama_bioactivity.csv"

print(f"Raíz del proyecto: {ROOT}")
print(f"Corpus PubChem: {CORPUS_CSV}")
print(f"Modo corpus: {CORPUS_MODE}")
print(f"Backend ChEMBL: {CHEMBL_BACKEND}")
print(f"Tipos de actividad: {len(STANDARD_TYPES)} ({', '.join(STANDARD_TYPES[:5])}…)")
print(f"Umbral activity_class: pChEMBL >= {PCHEMBL_ACTIVE_THRESHOLD}")

if CHEMBL_BACKEND == "sqlite":
    try:
        ensure_db_exists(CHEMBL_DB)
        info = db_info(CHEMBL_DB)
        print(f"ChEMBLdb: {CHEMBL_DB} ({info['db_size_bytes'] / 1e9:.2f} GB)")
    except FileNotFoundError as exc:
        print(f"[WARN] {exc}")
        print("  → docker compose -f docker/docker-compose.yml --profile setup run chembl-init")
else:
    print("Modo API REST — puede ser lento o fallar con errores 500 del servidor EBI.")

## 1. Carga del corpus

Por defecto (`corpus_mode=full`) se incluyen **todos** los compuestos de `pubchem_panama_cids.csv` con SMILES válido (~235 filas: MIDA + familias HID72).

Para volver al subconjunto de 20 MIDA: `CHEMBL_CORPUS_MODE=mida` o `corpus_mode: mida` en `config.yaml`.


In [ ]:
assert CORPUS_CSV.exists(), (
    f"No se encontró {CORPUS_CSV}. Ejecuta: make build-panama-corpus"
)

compounds_df = load_corpus_compounds(CORPUS_CSV, mode=CORPUS_MODE)

print(f"Modo corpus: {CORPUS_MODE}")
print(f"Compuestos cargados: {len(compounds_df)}")
if "is_mida" in compounds_df.columns:
    print(f"  → MIDA: {compounds_df['is_mida'].sum()}")
    print(f"  → Familias HID72 / otros: {(~compounds_df['is_mida']).sum()}")
print(f"Familias químicas: {compounds_df['family'].nunique()}")

invalid_smiles = compounds_df["smiles"].isna() | (compounds_df["smiles"].astype(str).str.strip() == "")
print(f"SMILES inválidos/vacíos: {invalid_smiles.sum()}")

display(compounds_df.head(10))
compounds_df.to_csv(COMPOUNDS_CSV, index=False)
print(f"\nGuardado: {COMPOUNDS_CSV}")


In [ ]:
fam_counts = compounds_df["family"].value_counts()
fig, ax = plt.subplots()
fam_counts.plot(kind="bar", ax=ax, color=sns.color_palette("muted", len(fam_counts)))
ax.set_title("20 ingredientes MIDA por familia química")
ax.set_xlabel("Familia")
ax.set_ylabel("Compuestos")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "mida_compounds_by_family.png", bbox_inches="tight")
plt.show()


## 2. Mapeo PubChem → ChEMBL

Estrategia en cascada: xref PubChem → SMILES → `pref_name` → sinónimo.
Si no hay match, se conserva `chembl_id = NaN` para trazabilidad.


In [ ]:
print(f"Resolviendo ChEMBL IDs (backend={CHEMBL_BACKEND})...\n")
mapping_df = build_mapping_table(
    compounds_df,
    backend=CHEMBL_BACKEND,
    db_path=CHEMBL_DB,
    verbose=True,
    existing_mapping_path=MAPPING_CSV,
    skip_resolved=True,
)

n_ok = (mapping_df["match_status"].isin(["ok", "ambiguous"])).sum()
n_nf = (mapping_df["match_status"] == "not_found").sum()

print(f"\nMatch OK/ambiguo: {n_ok} | No encontrados: {n_nf}")
display(mapping_df)
mapping_df.to_csv(MAPPING_CSV, index=False)
print(f"Guardado: {MAPPING_CSV}")


## 3. Descarga de bioactividad (raw)

Se descargan registros de **13 tipos de actividad** (IC50, EC50, Ki, Kd, Potency, Inhibition, AC50, LC50, GI50, MIC, LD50, ED50, IC90) sin aplicar filtros de calidad.

Los filtros (imputación pChEMBL, relación `=`, comentarios de invalidez) se aplican en la sección 5.


In [ ]:
print(f"Extrayendo bioactividad ChEMBL (backend={CHEMBL_BACKEND})...\n")
raw_df = build_bioactivity_table(
    mapping_df,
    backend=CHEMBL_BACKEND,
    db_path=CHEMBL_DB,
    verbose=True,
)

print(f"\nRegistros raw totales: {len(raw_df):,}")
if len(raw_df):
    display(raw_df.head(10))
    print("\nDistribución por tipo de medición:")
    display(raw_df["standard_type"].value_counts().to_frame("n"))
else:
    print("[WARN] No se descargaron actividades. Revisa conexión o mapeos ChEMBL.")

raw_df.to_csv(RAW_CSV, index=False)
print(f"\nGuardado: {RAW_CSV}")


## 4. Variable objetivo `activity_class`

- **Active:** `pchembl_value >= 6.0` (IC50 ≤ 1 µM)
- **Inactive:** `pchembl_value < 6.0`
- **NaN:** sin pChEMBL calculado


In [ ]:
if len(raw_df):
    class_counts = raw_df["activity_class"].value_counts(dropna=False)
    print("Distribución activity_class (dataset raw):")
    display(class_counts.to_frame("n"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    raw_df["pchembl_value"].dropna().hist(bins=40, ax=axes[0], color="#4C72B0", edgecolor="white")
    axes[0].axvline(PCHEMBL_ACTIVE_THRESHOLD, color="crimson", ls="--", label=f"Umbral {PCHEMBL_ACTIVE_THRESHOLD}")
    axes[0].set_title("Distribución pChEMBL (raw)")
    axes[0].set_xlabel("pChEMBL value")
    axes[0].legend()

    class_counts.plot(kind="bar", ax=axes[1], color=["#55A868", "#C44E52", "#CCCCCC"][: len(class_counts)])
    axes[1].set_title("Active / Inactive / NaN")
    axes[1].set_xlabel("")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "pchembl_and_activity_class_raw.png", bbox_inches="tight")
    plt.show()
else:
    print("Sin datos para visualizar.")


## 5. Filtros de calidad y trazabilidad

Reglas aplicadas al dataset de análisis (configurable en `config.yaml` → `chembl.quality_filters`):
1. **Imputar** `pchembl_value` desde `standard_value` (nM) cuando falte y `standard_relation = '='`
2. Excluir filas sin `pchembl_value` tras imputación
3. Excluir `standard_relation != '='`
4. Excluir `data_validity_comment` no nulo


In [ ]:
if len(raw_df):
    clean_df, filter_stats = apply_quality_filters_from_config(
        raw_df, ROOT / "config" / "config.yaml"
    )

    print(f"Registros raw:   {len(raw_df):,}")
    print(f"Registros clean: {len(clean_df):,}")
    print(f"Excluidos:       {len(raw_df) - len(clean_df):,}\n")

    display(filter_stats)
    clean_df.to_csv(CLEAN_CSV, index=False)
    print(f"\nGuardado: {CLEAN_CSV}")
else:
    clean_df = raw_df.copy()
    filter_stats = pd.DataFrame()
    print("Dataset vacío — no se aplicaron filtros.")


## 6. Resumen por compuesto


In [ ]:
summary_df = summarize_extraction(compounds_df, mapping_df, raw_df, clean_df)
display(summary_df)

if len(clean_df):
    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(summary_df))
    w = 0.35
    ax.bar([i - w/2 for i in x], summary_df["n_activities_raw"], width=w, label="Raw")
    ax.bar([i + w/2 for i in x], summary_df["n_activities_clean"], width=w, label="Clean")
    ax.set_xticks(list(x))
    ax.set_xticklabels(summary_df["compound_name"], rotation=45, ha="right")
    ax.set_ylabel("Nº actividades")
    ax.set_title("Actividades ChEMBL por plaguicida MIDA")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "activities_per_compound.png", bbox_inches="tight")
    plt.show()

print("\n=== Extracción completada ===")
print(f"  Compuestos MIDA:  {len(compounds_df)}")
print(f"  ChEMBL mapeados:  {mapping_df['chembl_id'].notna().sum()}")
print(f"  Actividades raw:  {len(raw_df):,}")
print(f"  Actividades clean:{len(clean_df):,}")


---
*Siguiente fase:* [`fase2_limpieza.ipynb`](fase2_limpieza.ipynb)
